# Nesri Discount — Automatisation de contenus Instagram

Ce notebook récupère le catalogue en direct depuis l'API Shopify de `nesridiscount.com`, sélectionne 4 produits du jour, génère les posts Instagram avec l'IA (Groq / Llama 3.3), crée les visuels, puis envoie un mail récapitulatif.

**Avant la première exécution :**
1. Crée un fichier `.env` à côté de ce notebook (voir cellule 1 ci-dessous, à exécuter UNE SEULE FOIS puis à ne plus exécuter).
2. Vérifie que `config.py` n'est pas nécessaire — tout est centralisé dans la cellule "CONFIGURATION" plus bas.


## 1. Configuration centrale

Toutes les variables sensibles (mots de passe, clés API) viennent du fichier `.env`.
Tous les chemins de fichiers sont **relatifs** au dossier du projet — donc cette configuration fonctionne sur n'importe quelle machine (Mac, Windows, Linux), pas seulement la tienne.

**Structure de dossiers attendue :**
```
Stage_Nesri_Discount/
├── .env                          ← identifiants (jamais partagé / jamais sur Git)
├── 02_Notebooks_Python/
│   └── test_shopify.ipynb       ← ce notebook
├── 03_Donnees_propres/
│   └── compteur_rotation.txt    ← créé automatiquement
└── assets/
    ├── logo_nesri.png
    ├── Arial.ttf                 ← (à fournir, voir cellule Police)
    └── Arial_Bold.ttf
```


In [5]:
# ============================================================
# 1. CONFIGURATION CENTRALE — identifiants + chemins relatifs
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

# Détecte automatiquement si on est dans 02_Notebooks_Python ou à la racine
if Path.cwd().name == "02_Notebooks_Python":
    BASE_DIR = Path.cwd().parent
    DOSSIER_NOTEBOOK = Path.cwd()
else:
    BASE_DIR = Path.cwd()
    DOSSIER_NOTEBOOK = Path.cwd()

# 🎯 CORRECTION RADICALE : On écrit le bon .env partout pour être sûr à 100%
contenu_propre = """GROQ_API_KEY=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
GMAIL_EXPEDITEUR=raadha.2910@gmail.com
GMAIL_MOT_DE_PASSE=eqjjzshldehtchbv
GMAIL_DESTINATAIRE=shivadheepi@gmail.com
"""

# On l'écrit à la racine ET dans le dossier du notebook
with open(BASE_DIR / ".env", "w") as f:
    f.write(contenu_propre)
with open(DOSSIER_NOTEBOOK / ".env", "w") as f:
    f.write(contenu_propre)

# On efface la mémoire cache bloquée de Jupyter
for cle_cache in ["GROQ_API_KEY", "GMAIL_MOT_DE_PASSE", "GMAIL_EXPEDITEUR", "GMAIL_DESTINATAIRE"]:
    if cle_cache in os.environ:
        del os.environ[cle_cache]

# Force le rechargement absolu du fichier qu'on vient de créer
load_dotenv(DOSSIER_NOTEBOOK / ".env", override=True)

# --- Identifiants ---
GMAIL_EXPEDITEUR   = os.getenv("GMAIL_EXPEDITEUR")
GMAIL_DESTINATAIRE = os.getenv("GMAIL_DESTINATAIRE")
GROQ_API_KEY       = os.getenv("GROQ_API_KEY")
GMAIL_MOT_DE_PASSE  = os.getenv("GMAIL_MOT_DE_PASSE")

DOSSIER_DONNEES = BASE_DIR / "03_Donnees_propres"
DOSSIER_ASSETS  = BASE_DIR / "assets"

CHEMIN_COMPTEUR = DOSSIER_DONNEES / "compteur_rotation.txt"
CHEMIN_LOGO     = DOSSIER_ASSETS / "logo_nesri.png"
CHEMIN_FONT_REGULAR = DOSSIER_ASSETS / "Arial.ttf"
CHEMIN_FONT_BOLD     = DOSSIER_ASSETS / "Arial_Bold.ttf"

DOSSIER_DONNEES.mkdir(parents=True, exist_ok=True)
DOSSIER_ASSETS.mkdir(parents=True, exist_ok=True)

# --- Affichage de validation ---
print("✅ Configuration chargée proprement depuis le fichier .env mis à jour :")
print(f"   - Gmail expéditeur   : {GMAIL_EXPEDITEUR}")
print(f"   - Gmail destinataire : {GMAIL_DESTINATAIRE}")
print(f"   - Clé Groq trouvée   : oui ({len(GROQ_API_KEY)} caractères, commence par {GROQ_API_KEY[:7]}...)")
print(f"   - Dossier données    : {DOSSIER_DONNEES}")
print(f"   - Dossier assets     : {DOSSIER_ASSETS}")

✅ Configuration chargée proprement depuis le fichier .env mis à jour :
   - Gmail expéditeur   : raadha.2910@gmail.com
   - Gmail destinataire : shivadheepi@gmail.com
   - Clé Groq trouvée   : oui (56 caractères, commence par gsk_BHD...)
   - Dossier données    : /Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres
   - Dossier assets     : /Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/assets


## 2. Récupération des produits en direct (API Shopify)

Si le site ne répond pas (panne, maintenance), le programme s'arrête proprement avec un message clair plutôt que de planter sans explication.


In [6]:
# ============================================================
# 2. RÉCUPÉRER TOUS LES PRODUITS (pagination avec pause + gestion d'erreurs)
# ============================================================

import requests
import pandas as pd
import re
import time

SHOP_URL = "https://nesridiscount.com"
MAX_TENTATIVES_429 = 5   # nombre de pauses max avant d'abandonner une page

tous_les_produits = []
page = 1
echec_definitif = False

while True:
    url = f"{SHOP_URL}/products.json?limit=250&page={page}"
    tentatives = 0

    while True:
        try:
            response = requests.get(url, timeout=15)
        except requests.exceptions.RequestException as e:
            print(f"🚨 Impossible de contacter {SHOP_URL} : {e}")
            echec_definitif = True
            break

        if response.status_code == 429:
            tentatives += 1
            if tentatives > MAX_TENTATIVES_429:
                print(f"🚨 Trop de 429 (rate limit) sur la page {page} — abandon.")
                echec_definitif = True
                break
            print(f"⏳ Rate limit atteint — pause 5 secondes (tentative {tentatives}/{MAX_TENTATIVES_429})...")
            time.sleep(5)
            continue

        if response.status_code != 200:
            print(f"❌ Erreur inattendue : {response.status_code}")
            echec_definitif = True
            break

        break  # réponse 200 reçue, on sort de la boucle de tentatives

    if echec_definitif:
        break

    produits_page = response.json().get("products", [])

    if not produits_page:
        break

    tous_les_produits.extend(produits_page)
    print(f"Page {page} → {len(produits_page)} produits récupérés")
    page += 1
    time.sleep(1)

if echec_definitif:
    print("\n🚨 RÉCUPÉRATION INTERROMPUE — le catalogue est incomplet ou vide.")
    print("   Le programme va s'arrêter ici pour éviter d'envoyer un mail avec des données fausses.")
    raise SystemExit("Arrêt : échec de récupération du catalogue Shopify.")

if len(tous_les_produits) == 0:
    print("\n🚨 Aucun produit récupéré — arrêt du programme.")
    raise SystemExit("Arrêt : catalogue vide.")

print(f"\n✅ Total récupéré : {len(tous_les_produits)} produits")


Page 1 → 250 produits récupérés
Page 2 → 250 produits récupérés
Page 3 → 250 produits récupérés
Page 4 → 250 produits récupérés
Page 5 → 250 produits récupérés
Page 6 → 250 produits récupérés
Page 7 → 250 produits récupérés
Page 8 → 250 produits récupérés
Page 9 → 250 produits récupérés
Page 10 → 250 produits récupérés
Page 11 → 250 produits récupérés
Page 12 → 250 produits récupérés
Page 13 → 250 produits récupérés
Page 14 → 250 produits récupérés
Page 15 → 250 produits récupérés
Page 16 → 250 produits récupérés
Page 17 → 250 produits récupérés
Page 18 → 250 produits récupérés
Page 19 → 250 produits récupérés
Page 20 → 60 produits récupérés

✅ Total récupéré : 4810 produits


## 3. Nettoyage des données et construction du tableau


In [7]:
# ============================================================
# 3. TRANSFORMER EN DATAFRAME PROPRE
# ============================================================

def clean_html(text):
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:600]

lignes = []

for produit in tous_les_produits:
    variante = produit["variants"][0] if produit["variants"] else {}
    image    = produit["images"][0]["src"] if produit["images"] else ""

    prix_vente = float(variante.get("price") or 0)
    prix_barre = variante.get("compare_at_price")
    prix_barre = float(prix_barre) if prix_barre else None

    # Ignorer les produits sans prix
    if prix_vente <= 0:
        continue

    economie   = round(prix_barre - prix_vente, 2) if prix_barre else None
    remise_pct = round((economie / prix_barre) * 100) if prix_barre else None

    lignes.append({
        "Handle"     : produit.get("handle", ""),
        "Titre"      : produit.get("title", ""),
        "Vendeur"    : produit.get("vendor", ""),
        "Prix_vente" : prix_vente,
        "Prix_barre" : prix_barre,
        "Economie"   : economie,
        "Remise_pct" : remise_pct,
        "Description": clean_html(produit.get("body_html", "")),
        "Image"      : image,
        "Type"       : produit.get("product_type", ""),
    })

df = pd.DataFrame(lignes)

# Supprimer les remises aberrantes (>70%)
nb_avant = len(df)
df = df[~(df["Remise_pct"] > 70)].copy()
df = df.reset_index(drop=True)

print(f"{'='*50}")
print(f"✅ CATALOGUE FINAL : {len(df)} produits propres")
print(f"   → {df['Prix_barre'].notna().sum()} produits EN PROMO")
print(f"   → {df['Prix_barre'].isna().sum()} produits à prix fixe")
print(f"   → Remises aberrantes supprimées : {nb_avant - len(df)}")
print(f"{'='*50}")
print(df[["Titre", "Prix_vente", "Prix_barre", "Remise_pct"]].head(10))


✅ CATALOGUE FINAL : 4807 produits propres
   → 2563 produits EN PROMO
   → 2244 produits à prix fixe
   → Remises aberrantes supprimées : 2
                                               Titre  Prix_vente  Prix_barre  \
0     Siège d'origine Campingaz Riviera – 5010001000        23.0         NaN   
1  Verre + diffuseur d'origine Campingaz Ambiance...        21.0         NaN   
2  Grille de cuisson fonte d'origine Campingaz Se...        49.0         NaN   
3  Piezo 1 sortie d'origine Campingaz Plancha / B...        12.0         NaN   
4  Parties latérales de cuve (droite + gauche) d'...        49.0         NaN   
5  Couvercle sans thermomètre d'origine Campingaz...        94.0         NaN   
6  Pare-vent d'origine Campingaz Plancha RL / Sig...        13.0         NaN   
7  Couvercle complet avec poignée, thermomètre et...        99.0         NaN   
8  Couvercle avec thermomètre d'origine Campingaz...        99.0         NaN   
9  Kit de mise à niveau de la ventilation pour ré...        

## 4. Sélection des 4 produits du jour

Logique de priorité : **Promo + Été** → **Promo seul** → **Été seul**.
Un compteur de rotation (sauvegardé dans `compteur_rotation.txt`) garantit qu'on ne republie pas deux fois le même produit deux jours de suite.

🔒 **Garde-fou ajouté** : si le pool de produits disponibles contient moins de 4 produits (cas extrême), le programme s'arrête avec un message clair au lieu de planter avec une erreur de division par zéro.


In [8]:
# ============================================================
# 4. SÉLECTION DES 4 PRODUITS DU JOUR (promo + saison été)
# ============================================================

# Mots-clés produits d'été
mots_cles_ete = [
    "ventilateur", "climatiseur", "climatisation", "rafraîchisseur", "rafraichisseur",
    "barbecue", "plancha", "glacière", "glaciere", "congélateur", "congelateur",
    "frigo", "réfrigérateur", "refrigerateur", "camping", "portabl",
    "piscine", "jardin", "terrasse", "extérieur", "exterieur"
]

def est_produit_ete(prod):
    texte = (str(prod['Titre']) + " " + str(prod['Description']) + " " + str(prod['Type'])).lower()
    return any(mot in texte for mot in mots_cles_ete)

# Créer les catégories
df['Est_ete']   = df.apply(est_produit_ete, axis=1)
df['Est_promo'] = df['Prix_barre'].notna()

df_promo_ete = df[df['Est_promo'] & df['Est_ete']].copy()
df_promo     = df[df['Est_promo'] & ~df['Est_ete']].copy()
df_ete       = df[~df['Est_promo'] & df['Est_ete']].copy()

print(f"📊 Répartition du catalogue :")
print(f"   🔥 Promo + Été  : {len(df_promo_ete)} produits")
print(f"   💰 Promo seul   : {len(df_promo)} produits")
print(f"   ☀️  Été seul     : {len(df_ete)} produits")

# Lire le compteur (chemin relatif désormais, voir cellule CONFIGURATION)
if not CHEMIN_COMPTEUR.exists():
    index_hier = -1
    print("\nPremier démarrage — compteur initialisé à 0")
else:
    with open(CHEMIN_COMPTEUR, 'r') as f:
        index_hier = int(f.read().strip())
    print(f"\nDernier index envoyé : {index_hier}")

# Construire le pool prioritaire : promo+été → promo → été
pool_prioritaire = pd.concat([df_promo_ete, df_promo, df_ete], ignore_index=True)
pool_prioritaire = pool_prioritaire.drop_duplicates(subset='Handle').reset_index(drop=True)

# 🔒 Garde-fou : on a besoin d'au moins 4 produits pour continuer
if len(pool_prioritaire) < 4:
    print(f"\n🚨 Seulement {len(pool_prioritaire)} produit(s) disponible(s) dans le pool — il en faut au moins 4.")
    raise SystemExit("Arrêt : pas assez de produits disponibles pour générer la sélection du jour.")

indices_aujourdhui = [(index_hier + i + 1) % len(pool_prioritaire) for i in range(4)]
liste_produits     = [pool_prioritaire.iloc[idx].copy() for idx in indices_aujourdhui]

# Nettoyage vendeur
for prod in liste_produits:
    if pd.isna(prod['Vendeur']) or str(prod['Vendeur']).strip() == '':
        prod['Vendeur'] = "Nesri Discount"

print(f"\n{'='*50}")
print(f"📦 4 PRODUITS DU JOUR (promo + saison été)")
print(f"{'='*50}")
for i, prod in enumerate(liste_produits, 1):
    tag = "🔥 PROMO+ÉTÉ" if prod['Est_promo'] and prod['Est_ete'] else "💰 PROMO" if prod['Est_promo'] else "☀️ ÉTÉ"
    if pd.notna(prod['Prix_barre']):
        prix_str = f"{int(prod['Prix_barre'])} € → {int(prod['Prix_vente'])} € (-{int(prod['Remise_pct'])}%)"
    else:
        prix_str = f"{int(prod['Prix_vente'])} €"
    print(f"[{i}/4] {tag} — {prod['Titre'][:45]}...")
    print(f"       💰 {prix_str}")
print(f"{'='*50}")


📊 Répartition du catalogue :
   🔥 Promo + Été  : 625 produits
   💰 Promo seul   : 1938 produits
   ☀️  Été seul     : 648 produits

Dernier index envoyé : 92

📦 4 PRODUITS DU JOUR (promo + saison été)
[1/4] 🔥 PROMO+ÉTÉ — Réfrigérateur – 130 L – 12/24V Indel B CRUISE...
       💰 1624 € → 1238 € (-24%)
[2/4] 🔥 PROMO+ÉTÉ — Globe Sphérique S Campingaz 37273EM...
       💰 19 € → 24 € (--26%)
[3/4] 🔥 PROMO+ÉTÉ — Réfrigérateur – 85 L – 12/24V Indel B CRUISE ...
       💰 1449 € → 1109 € (-23%)
[4/4] 🔥 PROMO+ÉTÉ — Frigocat 24V Indel B 90900012...
       💰 396 € → 269 € (-32%)


In [9]:
# ============================================================
# 5. RÉSUMÉ DES 4 PRODUITS DU JOUR
# ============================================================

print(f"📊 RÉSUMÉ DU CATALOGUE EN DIRECT :")
print(f"   → {len(df)} produits récupérés depuis nesridiscount.com")
print(f"   → {df['Prix_barre'].notna().sum()} en promo / {df['Prix_barre'].isna().sum()} à prix fixe")
print(f"   → Source : temps réel (API Shopify publique) ✅")
print(f"\n📦 4 produits sélectionnés aujourd'hui :")
for i, prod in enumerate(liste_produits, 1):
    print(f"   [{i}] {prod['Titre'][:55]}...")


📊 RÉSUMÉ DU CATALOGUE EN DIRECT :
   → 4807 produits récupérés depuis nesridiscount.com
   → 2563 en promo / 2244 à prix fixe
   → Source : temps réel (API Shopify publique) ✅

📦 4 produits sélectionnés aujourd'hui :
   [1] Réfrigérateur – 130 L – 12/24V Indel B CRUISE DR130...
   [2] Globe Sphérique S Campingaz 37273EM...
   [3] Réfrigérateur – 85 L – 12/24V Indel B CRUISE DR85...
   [4] Frigocat 24V Indel B 90900012...


## 5. Génération des posts Instagram (IA — Llama 3.3 via Groq)

Le client Groq utilise désormais la clé chargée dans la cellule **CONFIGURATION** — plus de clé en dur ici.


In [10]:
# ============================================================
# 6. GÉNÉRATION DES 4 POSTS INSTAGRAM VIA IA
# ============================================================

from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def format_prix(prix):
    if pd.isna(prix):
        return ""
    return f"{int(prix)} €"

def generer_post(prod):
    if pd.notna(prod['Prix_barre']):
        info_prix = (
            f"Prix original : {format_prix(prod['Prix_barre'])}\n"
            f"Prix actuel   : {format_prix(prod['Prix_vente'])}\n"
            f"Économie      : {format_prix(prod['Economie'])} soit -{int(prod['Remise_pct'])}% de remise"
        )
    else:
        info_prix = f"Prix : {format_prix(prod['Prix_vente'])}"

    prompt = f"""Tu es un expert en marketing digital pour une boutique en ligne française d'électroménager discount appelée Nesri Discount.

Génère un post Instagram accrocheur pour le produit suivant :

Nom du produit : {prod['Titre']}
Marque : {prod['Vendeur']}
{info_prix}
Description : {prod['Description'][:400]}

Règles importantes :
- Commence par une phrase d'accroche percutante avec des emojis
- Mets en valeur le prix et l'économie si le produit est en promo
- Les hashtags doivent être sans accents, pertinents au produit et à la marque, maximum 8
- Le ton doit être enthousiaste et dynamique comme un vrai community manager
- Le post doit faire entre 6 et 10 lignes avec des sauts de ligne entre chaque idée
- Écris en français
- Termine toujours par : 🛒 Disponible sur NesriDiscount.fr
"""
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.8
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"⚠️ Erreur IA pour '{prod['Titre'][:40]}...' : {e}")
        return f"🔥 Offre exceptionnelle sur {prod['Titre']} ! 🛒 Disponible sur NesriDiscount.fr"

# Générer les 4 posts
print("🤖 Génération des 4 posts Instagram...")
print("="*50)

posts_generes = []
for i, prod in enumerate(liste_produits, 1):
    print(f"\n📱 POST [{i}/4] — {prod['Titre'][:50]}...")
    post = generer_post(prod)
    posts_generes.append(post)
    print(post)
    print("-"*50)

print(f"\n✅ {len(posts_generes)} posts générés !")


🤖 Génération des 4 posts Instagram...

📱 POST [1/4] — Réfrigérateur – 130 L – 12/24V Indel B CRUISE DR13...
Voici un post Instagram accrocheur pour le produit :

"💥 Découvrez le réfrigérateur ultime pour vos aventures en camping-car, bateau ou installation solaire ! 🚣‍♂️🏕️

Économisez 386€ avec la remise de -24% sur le Réfrigérateur – 130 L – 12/24V Indel B CRUISE DR130 ! 🤑

Normallement à 1624€, vous pouvez l'acheter désormais à seulement 1238€ ! 💸

Ce réfrigérateur de grande capacité (130 litres) est conçu pour les environnements mobiles et autonomes. Alimenté en 12/24V DC, il est parfaitement adapté aux camping-cars, bateaux ou installations solaires. 🌞

Découvrez la fiabilité, le silence et l'écologie de ce super réfrigérateur ! 🌟

N'attendez plus pour profiter de cette offre exceptionnelle ! 🤩

Réfrigérateur – 130 L – 12/24V Indel B CRUISE DR130 : 1238€ (normalement 1624€) 📈

💡 Découvrez-le maintenant sur NesriDiscount.fr ! 🛒

#IndelB #Réfrigérateur #CampingCar #Bateau #Installati

## 6. Génération des visuels Instagram

**Corrections apportées :**
- Le chemin du logo est désormais relatif (`DOSSIER_ASSETS`), donc fonctionne sur n'importe quelle machine.
- La recherche de police fonctionne maintenant sur **Mac, Windows et Linux**, avec un message clair si aucune n'est trouvée (au lieu de basculer silencieusement vers une police par défaut illisible).
- Si tu veux une police garantie identique sur toutes les machines, place un fichier `Arial.ttf` et `Arial_Bold.ttf` (ou toute police libre comme **Roboto** ou **Open Sans**, téléchargeable sur [Google Fonts](https://fonts.google.com)) dans le dossier `assets/`.


In [13]:
# ============================================================
# 7. GÉNÉRATION DES 4 IMAGES EN MÉMOIRE — STYLE PREMIUM
# ============================================================

from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import requests as req
import random
import os
import pandas as pd

# ── POLICES DE SECOURS POUR GITHUB ACTIONS ──
_avertissement_police_affiche = False

def load_font(size, bold=False):
    global _avertissement_police_affiche
    
    # 1. On tente d'abord de charger tes fichiers Arial officiels depuis le dossier assets
    chemin_cible = CHEMIN_FONT_BOLD if bold else CHEMIN_FONT_REGULAR
    chemin_absolu = chemin_cible.resolve().as_posix()
    
    if os.path.exists(chemin_absolu):
        try:
            return ImageFont.truetype(chemin_absolu, size)
        except Exception as e:
            pass

    # 2. Si le fichier n'est pas trouvé ou plante sur Linux (GitHub), on utilise les polices natives Linux
    # Cela supprime définitivement les "petits carrés" et garde l'image superbe !
    polices_secours = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf" if bold else "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "LiberationSans-Bold.ttf" if bold else "LiberationSans-Regular.ttf"
    ]
    
    for p_secours in polices_secours:
        if os.path.exists(p_secours) or "/" not in p_secours:
            try:
                return ImageFont.truetype(p_secours, size)
            except:
                continue

    # 3. Sécurité absolue de PIL si tout échoue
    if not _avertissement_police_affiche:
        print(f"⚠️ Toutes les polices de caractères ont échoué — utilisation du secours par défaut.")
        _avertissement_police_affiche = True
    return ImageFont.load_default()

def text_width(draw, text, font):
    bbox = draw.textbbox((0, 0), text, font=font)
    return bbox[2] - bbox[0]

def fit_font(draw, text, max_width, start=40, min_size=22, bold=True):
    for size in range(start, min_size - 1, -2):
        font = load_font(size, bold=bold)
        if text_width(draw, text, font) <= max_width:
            return font
    return load_font(min_size, bold=bold)

def wrap_text(draw, text, font, max_width, max_lines=2):
    words = text.split()
    lines, current = [], ""
    for w in words:
        test = w if not current else current + " " + w
        if text_width(draw, test, font) <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = w
    if current:
        lines.append(current)
    return "\n".join(lines[:max_lines])

def generer_image_en_memoire(prod):
    W, H = 1080, 1350

    # ── FOND CLAIR CRÈME ÉLÉGANT ──
    canvas = Image.new("RGBA", (W, H))
    draw   = ImageDraw.Draw(canvas)
    for y in range(H):
        t = y / H
        r = int(250 + (238 - 250) * t)
        g = int(245 + (230 - 245) * t)
        b = int(235 + (210 - 235) * t)
        draw.line([(0, y), (W, y)], fill=(r, g, b, 255))

    # ── GRAIN SUBTIL ──
    random.seed(42)
    grain = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    gd    = ImageDraw.Draw(grain)
    for _ in range(12000):
        x  = random.randint(0, W)
        y  = random.randint(0, H)
        op = random.randint(3, 10)
        gd.point((x, y), fill=(180, 140, 40, op))
    canvas = Image.alpha_composite(canvas, grain)
    draw   = ImageDraw.Draw(canvas)

    # ── CERCLE DORÉ SUBTIL DERRIÈRE LE PRODUIT ──
    halo = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    hd   = ImageDraw.Draw(halo)
    hd.ellipse([W//2-330, 130, W//2+330, 830], fill=(212, 175, 55, 22))
    hd.ellipse([W//2-240, 190, W//2+240, 770], fill=(212, 175, 55, 14))
    canvas = Image.alpha_composite(canvas, halo)
    draw   = ImageDraw.Draw(canvas)

    # ── BORDURE DORÉE ──
    or_vif  = (180, 140, 40, 255)
    or_doux = (180, 140, 40, 80)
    draw.rectangle([28, 28, W-28, H-28], outline=or_vif,  width=2)
    draw.rectangle([34, 34, W-34, H-34], outline=or_doux, width=1)

    # ── COINS DÉCORÉS ──
    taille_coin = 30
    epaisseur   = 2
    coins = [(34, 34), (W-34, 34), (34, H-34), (W-34, H-34)]
    for (cx, cy) in coins:
        dx = 1 if cx < W//2 else -1
        dy = 1 if cy < H//2 else -1
        draw.line([(cx, cy), (cx + dx*taille_coin, cy)], fill=or_vif, width=epaisseur)
        draw.line([(cx, cy), (cx, cy + dy*taille_coin)], fill=or_vif, width=epaisseur)

    # ── FOND BLANC ARRONDI DERRIÈRE LE LOGO ──
    draw.rounded_rectangle([45, 42, 320, 42 + 90],
                            radius=12, fill=(255, 255, 255, 220))
    draw.rounded_rectangle([45, 42, 320, 42 + 90],
                            radius=12, outline=(180, 140, 40, 120), width=1)

    # ── LOGO ──
    logo_path = CHEMIN_LOGO.resolve().as_posix() if hasattr(CHEMIN_LOGO, "resolve") else str(CHEMIN_LOGO)
    if os.path.exists(logo_path):
        logo   = Image.open(logo_path).convert("RGBA")
        logo_w = 240
        logo_h = int(logo.height * (logo_w / logo.width))
        logo   = logo.resize((logo_w, logo_h))
        logo_y = 42 + (90 - logo_h) // 2
        canvas.alpha_composite(logo, (55, max(42, logo_y)))
    else:
        draw.text((60, 55), "NESRI DISCOUNT",
                  font=load_font(32, bold=True), fill=or_vif)

    draw = ImageDraw.Draw(canvas)

    # ── BADGE REMISE ──
    if pd.notna(prod['Prix_barre']) and int(prod['Remise_pct']) >= 10:
        bx, by = W - 185, 42
        draw.ellipse([bx-5, by-5, bx+155, by+155], outline=or_vif, width=2)
        draw.ellipse([bx, by, bx+150, by+150], fill=(180, 140, 40, 255))
        txt_rem = f"-{int(prod['Remise_pct'])}%"
        tw      = text_width(draw, txt_rem, load_font(40, bold=True))
        draw.text((bx + (150-tw)//2, by + 22),
                  txt_rem, font=load_font(40, bold=True), fill=(255, 255, 255, 255))
        draw.text((bx + 18, by + 78), "ECONOMIE",
                  font=load_font(17, bold=True), fill=(255, 255, 255, 220))

    # ── IMAGE PRODUIT ──
    image_url = prod.get("Image", "")
    if pd.notna(image_url) and str(image_url).strip() != "":
        try:
            res      = req.get(image_url, timeout=10)
            img_prod = Image.open(BytesIO(res.content)).convert("RGBA")
            box      = int(W * 0.65)
            img_prod.thumbnail((box, box))
            px = (W - img_prod.width) // 2
            py = int(H * 0.17)
            canvas.alpha_composite(img_prod, (px, py))
        except Exception as e:
            print(f"⚠️ Image produit non chargée : {e}")

    draw = ImageDraw.Draw(canvas)

    # ── SÉPARATEUR DORÉ ──
    sep_y = int(H * 0.615)
    for i, alpha in enumerate([40, 100, 180, 255, 180, 100, 40]):
        draw.line([(55 + i*2, sep_y), (W-55 - i*2, sep_y)],
                  fill=(180, 140, 40, alpha), width=1)

    # ── MARQUE ──
    marque_y = sep_y + 22
    draw.text((62, marque_y), str(prod['Vendeur']).upper(),
              font=load_font(22, bold=True), fill=(140, 100, 20, 255))

    # ── TITRE ──
    titre_y       = marque_y + 42
    titre_font    = fit_font(draw, prod['Titre'], W-120, start=36, min_size=20, bold=True)
    titre_wrapped = wrap_text(draw, prod['Titre'], titre_font, W-120, max_lines=2)
    draw.multiline_text((62, titre_y), titre_wrapped,
                        font=titre_font, fill=(30, 20, 10, 255), spacing=8)
    bbox_titre = draw.multiline_textbbox((62, titre_y), titre_wrapped,
                                          font=titre_font, spacing=8)
    prix_y = bbox_titre[3] + 26

    # ── PRIX VENTE ──
    txt_pv  = f"{int(prod['Prix_vente'])} €"
    font_pv = load_font(82, bold=True)
    draw.text((62, prix_y), txt_pv, font=font_pv, fill=(140, 100, 20, 255))

    # ── PRIX BARRÉ ──
    if pd.notna(prod['Prix_barre']):
        larg_pv = text_width(draw, txt_pv, font_pv)
        txt_pb  = f"{int(prod['Prix_barre'])} €"
        font_pb = load_font(26)
        next_x  = 62 + larg_pv + 22
        draw.text((next_x, prix_y + 46), txt_pb,
                  font=font_pb, fill=(160, 150, 130, 255))
        larg_pb = text_width(draw, txt_pb, font_pb)
        draw.line([(next_x, prix_y+61), (next_x+larg_pb, prix_y+61)],
                  fill=(200, 60, 60, 220), width=2)
        eco_txt = f"Vous economisez {int(prod['Economie'])} euros !"
        draw.text((62, prix_y + 100), eco_txt,
                  font=load_font(26, bold=True), fill=(140, 100, 20, 200))

    # ── BOUTON CTA ──
    cta_y = H - 148
    draw.rounded_rectangle([55, cta_y, W-55, cta_y+68],
                            radius=8, fill=(140, 100, 20, 255))
    draw.rounded_rectangle([55, cta_y, W-55, cta_y+68],
                            radius=8, outline=(200, 170, 80, 200), width=1)
    cta      = "VOIR L'OFFRE SUR NESRIDISCOUNT.FR"
    font_cta = load_font(24, bold=True)
    tw_cta   = text_width(draw, cta, font_cta)
    draw.text(((W - tw_cta) // 2, cta_y + 20),
              cta, font=font_cta, fill=(255, 255, 255, 255))

    # ── LIGNE DORÉE BAS ──
    draw.rectangle([28, H-31, W-28, H-28], fill=(180, 140, 40, 255))

    buffer = BytesIO()
    canvas.convert("RGB").save(buffer, format="PNG")
    buffer.seek(0)
    return buffer.read()


# ── GÉNÉRER LES 4 IMAGES ──
print("🎨 Génération des 4 images en mémoire...")
images_bytes    = []
images_en_echec = []

for i, prod in enumerate(liste_produits, 1):
    try:
        img = generer_image_en_memoire(prod)
        images_bytes.append(img)
        print(f"   ✅ Image [{i}/4] générée ({len(img) // 1024} Ko)")
    except Exception as e:
        print(f"   🚨 Image [{i}/4] ÉCHEC : {e}")
        images_en_echec.append(i)

print(f"\n✅ {len(images_bytes)} images générées !")

🎨 Génération des 4 images en mémoire...
   ✅ Image [1/4] générée (178 Ko)
   ✅ Image [2/4] générée (132 Ko)
   ✅ Image [3/4] générée (176 Ko)
   ✅ Image [4/4] générée (169 Ko)

✅ 4 images générées !


## 7. Envoi du mail récapitulatif

**Corrections apportées :**
- Identifiants Gmail lus depuis la config centrale (plus de mot de passe en dur).
- Le mail n'est envoyé **que si les 4 images ont bien été générées** (sécurité ajoutée par rapport à la version précédente).


In [14]:
# ============================================================
# 8. ENVOI PAR MAIL — 4 PRODUITS + 4 IMAGES EN PIÈCES JOINTES
# ============================================================

import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import pandas as pd

# ── DÉFINITION DE LA FONCTION DE NETTOYAGE POUR GITHUB ACTIONS ──
def nettoyer_texte(texte):
    if not texte or pd.isna(texte):
        return ""
    # Remplace les espaces insécables et nettoie les espaces inutiles
    return str(texte).replace('\xa0', ' ').strip()

# 🔒 Garde-fou : on n'envoie le mail que si on a bien 4 images ET 4 posts
if len(images_bytes) != 4 or len(posts_generes) != 4:
    print(f"🚨 Envoi annulé : {len(images_bytes)} image(s) et {len(posts_generes)} post(s) au lieu de 4 chacun.")
    print("   Corrige le problème dans les cellules précédentes avant de réessayer.")
else:
    expediteur   = GMAIL_EXPEDITEUR
    mot_de_passe = GMAIL_MOT_DE_PASSE
    destinataire = GMAIL_DESTINATAIRE

    # Nettoyer les caractères problématiques (espaces insécables, etc.) dans tous les produits
    for prod in liste_produits:
        prod['Titre'] = nettoyer_texte(prod['Titre'])
        prod['Description'] = nettoyer_texte(prod['Description'])
        prod['Vendeur'] = nettoyer_texte(prod['Vendeur'])

    posts_generes = [nettoyer_texte(post) for post in posts_generes]

    # Construire les blocs HTML pour les 4 produits
    blocs_html = ""
    for i, (prod, post) in enumerate(zip(liste_produits, posts_generes), 1):
        if pd.notna(prod['Prix_barre']):
            badge     = f'<span style="background:#E30613;color:#fff;font-size:12px;font-weight:bold;padding:3px 8px;border-radius:4px;margin-left:8px;">-{int(prod["Remise_pct"])}%</span>'
            prix_barre_html = f'<span style="color:#999;text-decoration:line-through;margin-right:8px;font-size:14px;">{int(prod["Prix_barre"])} €</span>'
        else:
            badge           = ""
            prix_barre_html = ""

        blocs_html += f"""
        <div style="border:1px solid #eee;border-radius:8px;padding:16px;margin-bottom:20px;background:#fafafa;">
          <table width="100%" cellspacing="0" cellpadding="0">
            <tr>
              <td width="30%" align="center">
                <img src="{prod['Image']}" style="max-width:120px;max-height:120px;object-fit:contain;" />
              </td>
              <td width="70%" style="padding-left:15px;">
                <p style="margin:0 0 4px 0;font-size:11px;color:#888;">PRODUIT {i}/4</p>
                <h3 style="margin:0 0 6px 0;color:#1a1a1a;font-size:15px;">{prod['Titre']}</h3>
                <p style="margin:0 0 8px 0;color:#E30613;font-weight:bold;font-size:18px;">
                  {prix_barre_html}{int(prod['Prix_vente'])} € {badge}
                </p>
                <p style="margin:0 0 6px 0;font-size:11px;color:#888;">📎 Visuel Instagram : post_instagram_{i}.png</p>
                <p style="margin:0;font-size:12px;color:#444;line-height:1.5;
                          white-space:pre-line;font-style:italic;">{post}</p>
              </td>
            </tr>
          </table>
        </div>
        """

    html = f"""
    <!DOCTYPE html>
    <html>
    <body style="margin:0;padding:0;background:#f4f4f4;font-family:Arial,sans-serif;">
    <div style="max-width:600px;margin:20px auto;background:#fff;border-radius:12px;
                overflow:hidden;box-shadow:0 4px 10px rgba(0,0,0,0.1);">

      <div style="background:#0c1636;padding:24px;text-align:center;">
        <h1 style="color:#fff;font-size:26px;margin:0;letter-spacing:1px;">NESRI DISCOUNT</h1>
        <p style="color:#00f59c;font-size:14px;margin:5px 0 0;font-weight:bold;">
          🔥 SÉLECTION DU JOUR — 4 PRODUITS
        </p>
      </div>

      <div style="padding:20px 25px;">
        <p style="font-size:15px;color:#333;">Bonjour équipe communication,</p>
        <p style="font-size:14px;color:#666;margin-bottom:20px;">
          Voici votre sélection automatisée de 4 produits du jour avec leurs posts Instagram
          et visuels prêts à publier (pièces jointes).
        </p>

        {blocs_html}

        <div style="text-align:center;margin-top:25px;">
          <a href="https://nesridiscount.fr"
             style="background:#E30613;color:#fff;text-decoration:none;
                    padding:12px 24px;border-radius:6px;font-weight:bold;display:inline-block;">
            Accéder au site officiel
          </a>
        </div>
      </div>

    </div>
    </body>
    </html>
    """

    # Construire le mail
    message            = MIMEMultipart("mixed")
    message["From"]    = expediteur
    message["To"]      = destinataire
    message["Subject"] = "🔥 Sélection du jour — 4 produits Nesri Discount"

    msg_html = MIMEMultipart("alternative")
    msg_html.attach(MIMEText(html, "html"))
    message.attach(msg_html)

    for i, img_bytes in enumerate(images_bytes, 1):
        piece = MIMEBase("application", "octet-stream")
        piece.set_payload(img_bytes)
        encoders.encode_base64(piece)
        piece.add_header("Content-Disposition", f'attachment; filename="post_instagram_{i}.png"')
        message.attach(piece)

    # Envoi
    try:
        print("🚀 Envoi du mail avec 4 produits + 4 images...")
        with smtplib.SMTP("smtp.gmail.com", 587) as serveur:
            serveur.ehlo()
            serveur.starttls()
            serveur.ehlo()
            serveur.login(expediteur, mot_de_passe)
            serveur.send_message(message)
        print("🎉 Mail envoyé avec succès !")
        # Mettre à jour le compteur (chemin relatif désormais)
        with open(CHEMIN_COMPTEUR, 'w') as f:
            f.write(str(indices_aujourdhui[-1]))
        print(f"✅ Compteur mis à jour → demain on repart à l'index {indices_aujourdhui[-1] + 1}")

    except Exception as e:
        print(f"🚨 Erreur lors de l'envoi : {e}")
        print("   → Le compteur n'a pas été mis à jour, les mêmes produits seront retentés demain.")

🚀 Envoi du mail avec 4 produits + 4 images...
🎉 Mail envoyé avec succès !
✅ Compteur mis à jour → demain on repart à l'index 97
